# Setup

In [1]:
from pyiron import Project
import os
import numpy as np
import time
from IPython.display import clear_output, display
from datetime import datetime

In [2]:
# Global Variables
VaspExecutable = "6.4.3_std_mpi"

# Useful Modules

In [3]:
def CreateStrainRange(ZeroStrainPoint_Percent:float = 0.0, Range_Percent:float = 0.1, NumberOfStrains: int = 5, Regime:str = "even"):
    StrainRange = []
    if Regime == "even":
        # Ensure that the strain distribution is centered around 0% strain
        if NumberOfStrains % 2 == 0:
            NumberOfStrains += 1
        
        StepSize = Range_Percent/int(NumberOfStrains/2)
        OneSidedStrain = [Strain.item() for Strain in np.linspace(ZeroStrainPoint_Percent+StepSize, Range_Percent, num=int(NumberOfStrains/2))]
        
        for Strain in reversed(OneSidedStrain):
            StrainRange.append(-1*Strain)
        StrainRange.append(ZeroStrainPoint_Percent)
        for Strain in OneSidedStrain:
            StrainRange.append(Strain)
    return StrainRange

In [4]:
# Creates a log file, to dump info during runtime
class LoggingFile():
    def __init__(self, FilePath: str, Mode: str = "w"):
        self.FilePath = FilePath
        self.File = open(FilePath, Mode)

    def LogEvent(self, Event, NewLine: bool = True):
        self.File.write(f"{datetime.now():%H:%M:%S} | {Event}" + ("\n" if NewLine else ""))
        self.File.flush()

    
    def LogInformation(self, Input, NewLine: bool = True):
        self.File.write(str(Input) + "\n" if NewLine else str(Input))
        self.File.flush()

    def Close(self):
        self.File.close()

    def __del__(self):
        if not self.File.closed:
            self.File.close()

# Calculations
## Porject definition

In [5]:
SimulationFilePath = "SimulationFileDump"

In [6]:
if "Log" in globals():
    Log.Close()
Log = LoggingFile("ElasticConstants.log")

Log.LogEvent("Start bulk modulus calculation\n")

ElasticConstantsProject = Project(path=SimulationFilePath)
ElasticConstantsProject.remove_jobs(recursive=True, silently=True)

  0%|          | 0/1 [00:00<?, ?it/s]

In [7]:
UnitCell = ElasticConstantsProject.create.structure.ase.bulk("Al", cubic=True, a = 2.75)
#UnitCell.plot3d()

## Initial Relaxation

To apply accurate strains, the structure needs to be in the minimum energy state. This is achieved by first relaxing the structure with VASP, reading the resulting lattice constant and the manually searching for the minimum, by applying small uniform strains

In [8]:
# Create relaxation Job
InitialRelaxationJob = ElasticConstantsProject.create_job(job_type=ElasticConstantsProject.job_type.Vasp, job_name="InitialRelaxation", delete_existing_job=True)
InitialRelaxationJob.server.queue = "small"
InitialRelaxationJob.server.cores = 4

InitialRelaxationJob.structure = UnitCell.copy()

InitialRelaxationJob.input.incar["ENCUT"] = 650
InitialRelaxationJob.input.incar["IBRION"] = 2
InitialRelaxationJob.input.incar["ISIF"] = 3
InitialRelaxationJob.input.incar["NSW"] = 200

Log.LogInformation(InitialRelaxationJob.input.incar)

InitialRelaxationJob.executable = VaspExecutable
InitialRelaxationJob.run()

The job InitialRelaxation was saved and received the ID: 38
Queue system id:  5891503


In [9]:
while not all(ElasticConstantsProject.job_table()["status"].isin(["finished", "aborted"])):
    clear_output(wait=True)
    display(ElasticConstantsProject.job_table())
    time.sleep(10)

clear_output(wait=True)
display(ElasticConstantsProject.job_table())

,id,status,chemicalformula,job,subjob,projectpath,project,timestart,timestop,totalcputime,computer,hamilton,hamversion,parentid,masterid
0,38,finished,Al4,InitialRelaxation,/InitialRelaxation,None,/scratch/htc/jhornig/01_PyIron/PyIronWorkflows/src/Workflows/ElasticConstants/SimulationFileDump/,2026-11-04 09:24:21.723590,2026-11-04 09:24:30.496643,8.0,pyiron@htc-login03.zib.de#4#small,Vasp,6.4.3_std_mpi,None,None


In [13]:
# Read relaxed structure
RelaxedStructure = InitialRelaxationJob.get_structure(iteration_step=-1)

print(RelaxedStructure)

LatticeParameter_A = RelaxedStructure.cell[0][0]
Volume_A3 = RelaxedStructure.get_volume()
print(LatticeParameter_A, Volume_A3)

/tmp/ipykernel_2196018/2179723050.py:2: DeprecationWarning: pyiron_atomistics.atomistics.structure.has_structure.HasStructure.get_structure(iteration_step=-1) is deprecated.
  RelaxedStructure = InitialRelaxationJob.get_structure(iteration_step=-1)
2026-11-04 09:30:12,174 - pyiron_log - WARNING - Could not access indices, returning None!
2026-11-04 09:30:12,198 - pyiron_log - WARNING - Could not access indices, returning None!


Al4
4.03715122 65.79987239437216


In [10]:
MaximumStrainMagnitude_Percent = 0.1
StrainMagnitudes = CreateStrainRange(ZeroStrainPoint_Percent=0, Range_Percent=MaximumStrainMagnitude_Percent, NumberOfStrains=15)

In [11]:
for Strain in StrainMagnitudes:
    

SyntaxError: incomplete input (3883366898.py, line 2)